# 02 — Pré-processamento e Engenharia de Atributos

**Objetivo:** Limpar o dataset, corrigir tipos de dados, criar novas variáveis
(ex.: tempo de espera em dias, faixas etárias), tratar o desbalanceamento de classes
e gerar o conjunto final de treino/teste pronto para modelagem.

## 1. Limpeza (issue #7)

As funções de limpeza foram implementadas em `src/data.py` (issue #5, PR #34),
seguindo os problemas mapeados no [dicionário de dados](../docs/dicionario_dados.md):

- `rename_columns`: nomes em snake_case (`Hipertension` → `hypertension`, `No-show` → `no_show`, etc.).
- `convert_types`: `scheduled_day`/`appointment_day` → datetime UTC; `patient_id` → string
  (os 5 IDs com decimais, ex.: `93779.52927`, são preservados para não colidir com IDs reais);
  `no_show` → 1/0 (1 = faltou), idempotente.
- `remove_invalid`: remove `age < 0` (1 linha) e consultas cuja data é anterior ao
  agendamento (5 linhas). Idades acima de 100 são **mantidas** — decisão registrada
  em `docs/decisoes.md`.
- `drop_duplicates`: remove repetições de todas as colunas exceto `appointment_id` e
  `no_show`, mantendo a primeira ocorrência (632 linhas, incluindo os 14 grupos com
  alvo divergente).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import data, features

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

raw = data.load_raw('../data/raw/KaggleV2-May-2016.csv')

In [ ]:
df = data.clean(raw)

In [ ]:
# Detalhamento das remoções por regra
r = data.convert_types(data.rename_columns(raw))
invalido = (r['age'] < 0) | (r['appointment_day'].dt.normalize() < r['scheduled_day'].dt.normalize())
r2 = r[~invalido]
chave = [c for c in r2.columns if c not in ('appointment_id', 'no_show')]
divergente = r2.groupby(chave)['no_show'].transform('nunique') > 1
dup = r2.duplicated(subset=chave)

pd.Series({
    'idade negativa': int((r['age'] < 0).sum()),
    'consulta antes do agendamento': int((r['appointment_day'].dt.normalize() < r['scheduled_day'].dt.normalize()).sum()),
    'duplicata idêntica (mesmo alvo)': int((dup & ~divergente).sum()),
    'duplicata com alvo divergente (mantém 1ª)': int((dup & divergente).sum()),
    'total removido': int(invalido.sum() + dup.sum()),
})

In [ ]:
df.dtypes

In [ ]:
df.head()

## 2. Engenharia de atributos (issue #7)

`src/features.py::build_features` cria três atributos:

- `waiting_days`: dias entre a data do agendamento e a data da consulta.
  Na EDA, agendamentos para o próprio dia faltam 4,6%, contra 28,5% com espera —
  tende a ser o preditor mais forte.
- `weekday`: dia da semana da consulta (0 = segunda, 6 = domingo). Não há consultas
  em domingo e quase nenhuma em sábado.
- `same_day`: 1 se a consulta é no mesmo dia do agendamento. É redundante com
  `waiting_days == 0`; fica para a modelagem decidir se mantém (PR #34).

In [ ]:
feat = features.build_features(df)
print(feat.shape)
feat[['waiting_days', 'weekday', 'same_day']].describe()

In [ ]:
# Sanity check das novas colunas
assert feat['waiting_days'].min() >= 0
assert (feat['same_day'] == (feat['waiting_days'] == 0).astype(int)).all()
print('Agendamentos no mesmo dia:', int(feat['same_day'].sum()))
print('Taxa de falta (mesmo dia):', round(feat.loc[feat['same_day'] == 1, 'no_show'].mean(), 3))
print('Taxa de falta (com espera):', round(feat.loc[feat['same_day'] == 0, 'no_show'].mean(), 3))

dias = ['seg', 'ter', 'qua', 'qui', 'sex', 'sáb', 'dom']
taxa = feat.groupby('weekday', observed=True)['no_show'].agg(['mean', 'count'])
taxa.index = [dias[i] for i in taxa.index]
taxa

In [ ]:
# Taxa de falta sobe rápido com os primeiros dias de espera
fig, ax = plt.subplots(figsize=(8, 3))
faixas = pd.cut(feat['waiting_days'], bins=[-1, 0, 2, 7, 14, 30, 60, 200],
                labels=['0', '1–2', '3–7', '8–14', '15–30', '31–60', '60+'])
taxa_espera = feat.groupby(faixas, observed=True)['no_show'].mean()
sns.barplot(x=taxa_espera.index, y=taxa_espera.values, ax=ax, color='steelblue')
ax.set(title='Taxa de falta por dias de espera', xlabel='Dias de espera', ylabel='Taxa de falta')
plt.show()

## 3. Exportação

O dataset limpo com os atributos vai para `data/processed/consultas_limpas.csv`
(pasta ignorada no git) — o mesmo artefato gerado por `python -m src.data`.

**Atenção ao reler o CSV:** os tipos não sobrevivem ao roundtrip (`patient_id` volta
como `float64`, datas voltam como texto). É preciso rodar `data.convert_types()`
de novo, que é idempotente para `no_show` já numérico — ver `docs/decisoes.md`.

In [ ]:
data.save_processed(feat, '../data/processed/consultas_limpas.csv')

re = pd.read_csv('../data/processed/consultas_limpas.csv')
print('Tipos ao reler:', re['patient_id'].dtype, '/', re['scheduled_day'].dtype)
re = data.convert_types(re)
print('Após convert_types:', re['patient_id'].dtype, '/', re['scheduled_day'].dtype)